In [ ]:
# Scary Barbie, step 1: which NEOWISE images cover it, and when?
import numpy as np
from astropy.table import Table
from astropy.time import Time
from astropy.coordinates import SkyCoord
from astropy.utils.data import conf

conf.remote_timeout = 300                                # give IRSA up to 5 minutes
barbie = SkyCoord("21h13m48.415s", "+27d25m50.38s")      # AT2021lwx

url = ("https://irsa.ipac.caltech.edu/ibe/search/wise/neowiser/p1bm_frm"
       f"?POS={barbie.ra.deg},{barbie.dec.deg}")
frames = Table.read(url, format="ipac")
tcol = "mjd" if "mjd" in frames.colnames else [c for c in frames.colnames if "mjd" in c.lower()][0]
frames.sort(tcol)
print(len(frames), "NEOWISE images cover Scary Barbie")

# Group the images into visits: a gap of more than 30 days starts a new visit
visit = np.concatenate([[0], np.cumsum(np.diff(frames[tcol]) > 30)])
frames["visit"] = visit
for v in np.unique(visit):
    f = frames[visit == v]
    day = Time(np.median(f[tcol]), format="mjd").iso[:10]
    print(f"visit {v:2d}  {day}   W1: {np.sum(f['band'] == 1):3d}   W2: {np.sum(f['band'] == 2):3d}")

In [ ]:
# Scary Barbie, step 2: download cutouts, stack each visit, save the stacks
!pip install -q reproject
import os
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.stats import sigma_clipped_stats
from astropy.utils.data import download_file
from reproject import reproject_interp

# a 2' x 2' grid centred on Scary Barbie, using WISE's own pixel size (2.75")
grid = WCS(naxis=2)
grid.wcs.ctype = ["RA---TAN", "DEC--TAN"]
grid.wcs.crval = [barbie.ra.deg, barbie.dec.deg]
grid.wcs.crpix = [23, 23]
grid.wcs.cdelt = [-2.75 / 3600, 2.75 / 3600]
shape = (45, 45)
ZP = {1: 20.5, 2: 19.5}     # common brightness scale every image is put on (Vega mag)

def cutout_url(row):
    scan, num, band = str(row["scan_id"]), int(row["frame_num"]), int(row["band"])
    grp = str(row["scangrp"]) if "scangrp" in frames.colnames else scan[-2:]
    return ("https://irsa.ipac.caltech.edu/ibe/data/wise/neowiser/p1bm_frm/"
            f"{grp}/{scan}/{num:03d}/{scan}{num:03d}-w{band}-int-1b.fits"
            f"?center={barbie.ra.deg},{barbie.dec.deg}&size=150arcsec&gzip=false")

# skip images NEOWISE flags as poor quality (only if those columns exist)
good = np.ones(len(frames), bool)
for col in ["qual_frame", "qi_fact", "saa_sep"]:
    if col in frames.colnames:
        good &= np.array(frames[col]) > 0

os.makedirs("/content/barbie_stacks", exist_ok=True)
stacks, last_err = {}, None
for v in np.unique(frames["visit"]):
    msg = []
    for band in (1, 2):
        rows = frames[good & (frames["visit"] == v) & (frames["band"] == band)]
        imgs, n_bad = [], 0
        for row in rows:
            try:
                with fits.open(download_file(cutout_url(row), cache=True)) as hdul:
                    data, hdr = hdul[0].data.astype(float), hdul[0].header
                arr, _ = reproject_interp((data, WCS(hdr)), grid, shape_out=shape)
                _, bkg, _ = sigma_clipped_stats(arr)
                imgs.append((arr - bkg) * 10 ** (-0.4 * (hdr["MAGZP"] - ZP[band])))
            except Exception as err:
                n_bad, last_err = n_bad + 1, repr(err)[:100]
        if imgs:
            stacks[(v, band)] = np.nanmedian(imgs, axis=0)
            out = grid.to_header()
            out["MJD"], out["NIMAGES"], out["MAGZP"] = float(np.median(rows[tcol])), len(imgs), ZP[band]
            fits.writeto(f"/content/barbie_stacks/visit{v:02d}_w{band}.fits",
                         stacks[(v, band)], out, overwrite=True)
        msg.append(f"W{band}: {len(imgs)} used, {n_bad} skipped")
    print(f"visit {v:2d}   " + "   ".join(msg))
if last_err:
    print("last error:", last_err)

In [ ]:
show = {10: "2019 (before)", 14: "2021 (peak)", 17: "2022", 20: "2024 (last)"}
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for r, band in enumerate((1, 2)):
    lo, hi = np.nanpercentile(stacks[(14, band)], [5, 99.5])   # same scale for the whole row
    for c, (v, label) in enumerate(show.items()):
        ax = axes[r, c]
        ax.imshow(stacks[(v, band)], origin="lower", cmap="gray", vmin=lo, vmax=hi)
        ax.add_patch(plt.Circle((22, 22), 3, fill=False, color="red"))
        ax.set_title(f"W{band}  {label}", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# Scary Barbie, step 3a: subtract the "before" image (2014-2019) from every visit
template = {b: np.nanmedian([stacks[(v, b)] for v in range(12) if (v, b) in stacks], axis=0)
            for b in (1, 2)}
diff = {(v, b): stacks[(v, b)] - template[b] for (v, b) in stacks}

# where is the bright neighbour? (brightest spot within 10 pixels of Barbie)
yy, xx = np.indices(shape)
near = np.where(np.hypot(xx - 22, yy - 22) < 10, template[1], -np.inf)
sy, sx = np.unravel_index(np.nanargmax(near), shape)
print(f"bright star is {np.hypot(sx - 22, sy - 22) * 2.75:.0f} arcsec from Scary Barbie")

show = {10: "2019 (before)", 14: "2021 (peak)", 17: "2022", 20: "2024 (last)"}
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for r, b in enumerate((1, 2)):
    _, _, noise = sigma_clipped_stats(diff[(10, b)])
    for c, (v, label) in enumerate(show.items()):
        ax = axes[r, c]
        ax.imshow(diff[(v, b)], origin="lower", cmap="gray", vmin=-4 * noise, vmax=8 * noise)
        ax.add_patch(plt.Circle((22, 22), 3, fill=False, color="red"))
        ax.set_title(f"W{b}  {label}  minus before", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
# Scary Barbie, step 3b: measure its brightness in every visit
def ap_sum(img, x, y, r=2.5):
    return np.nansum(img[np.hypot(xx - x, yy - y) <= r])

# empty spots away from Barbie and the bright star, to measure the noise
spots = [(x, y) for x in range(5, 41, 4) for y in range(5, 41, 4)
         if np.hypot(x - 22, y - 22) > 7 and np.hypot(x - sx, y - sy) > 7]

out = []
for (v, b) in sorted(diff):
    flux = ap_sum(diff[(v, b)], 22, 22)
    _, _, err = sigma_clipped_stats([ap_sum(diff[(v, b)], x, y) for x, y in spots])
    mjd = float(np.median(frames[tcol][frames["visit"] == v]))
    out.append((v, b, mjd, flux, err))
lc = Table(rows=out, names=("visit", "band", "mjd", "flux", "err"))
lc.write("/content/barbie_lightcurve_raw.csv", overwrite=True)

plt.figure(figsize=(9, 5))
for b, colour in ((1, "tab:blue"), (2, "tab:red")):
    s = lc[lc["band"] == b]
    plt.errorbar(Time(s["mjd"], format="mjd").decimalyear, s["flux"], s["err"],
                 fmt="o", color=colour, label=f"W{b}")
plt.axhline(0, color="grey", lw=0.8)
plt.axvline(2020.46, color="k", ls=":", label="optical flare starts (June 2020)")
plt.xlabel("Year")
plt.ylabel("Brightness above the 'before' level (counts)")
plt.legend()
plt.show()

In [ ]:
# Scary Barbie, step 3c: separate "before" images for May and November visits
pre = range(12)                      # visits 0-11: 2014-2019, before the flare

def template_for(v, b):
    """Median of the 'before' visits from the same season (May or November), leaving out v."""
    use = [stacks[(u, b)] for u in pre
           if u % 2 == v % 2 and u != v and (u, b) in stacks]   # even = May, odd = November
    return np.nanmedian(use, axis=0)

diff2 = {(v, b): stacks[(v, b)] - template_for(v, b) for (v, b) in stacks}

out = []
for (v, b) in sorted(diff2):
    flux = ap_sum(diff2[(v, b)], 22, 22)
    _, _, err = sigma_clipped_stats([ap_sum(diff2[(v, b)], x, y) for x, y in spots])
    mjd = float(np.median(frames[tcol][frames["visit"] == v]))
    out.append((v, b, mjd, flux, err))
lc2 = Table(rows=out, names=("visit", "band", "mjd", "flux", "err"))
lc2.write("/content/barbie_lightcurve_v2.csv", overwrite=True)

# how much the 'before' points really scatter, compared with the error bars
for b in (1, 2):
    s = lc2[(lc2["band"] == b) & (lc2["visit"] < 12)]
    print(f"W{b}: before-flare scatter = {np.std(s['flux']):.1f} counts, "
          f"typical error bar = {np.median(s['err']):.1f} counts")

plt.figure(figsize=(9, 5))
for b, colour in ((1, "tab:blue"), (2, "tab:red")):
    s = lc2[lc2["band"] == b]
    plt.errorbar(Time(s["mjd"], format="mjd").decimalyear, s["flux"], s["err"],
                 fmt="o", color=colour, label=f"W{b}")
plt.axhline(0, color="grey", lw=0.8)
plt.axvline(2020.46, color="k", ls=":", label="optical flare starts (June 2020)")
plt.xlabel("Year")
plt.ylabel("Brightness above the 'before' level (counts)")
plt.legend()
plt.show()

In [ ]:
# Scary Barbie, step 4a: put the light curve in real units, using catalogue stars
from urllib.parse import quote

def irsa_query(adql):
    url = "https://irsa.ipac.caltech.edu/TAP/sync?QUERY=" + quote(adql)
    t = Table.read(download_file(url, cache=False), format="votable", use_names_over_ids=True)
    for n in t.colnames:
        if n != n.lower():
            t.rename_column(n, n.lower())
    print(len(t), "rows, columns:", t.colnames)
    return t

circle = (f"CONTAINS(POINT('J2000', ra, dec), "
          f"CIRCLE('J2000', {barbie.ra.deg}, {barbie.dec.deg}, 0.03)) = 1")
everything = irsa_query(f"SELECT ra, dec FROM allwise_p3as_psd WHERE {circle}")
cat = irsa_query(f"SELECT ra, dec, w1mpro, w2mpro FROM allwise_p3as_psd WHERE {circle} "
                 "AND w1mpro BETWEEN 9 AND 16 AND w2mpro IS NOT NULL "
                 "AND ph_qual LIKE 'AA%' AND cc_flags LIKE '00%'")

# keep stars well inside the image with no other source within 12"
pos = SkyCoord(cat["ra"], cat["dec"], unit="deg")
_, gap, _ = pos.match_to_catalog_sky(SkyCoord(everything["ra"], everything["dec"], unit="deg"),
                                     nthneighbor=2)
cx, cy = grid.world_to_pixel(pos)
use = (gap.arcsec > 8) & (cx > 4) & (cx < 40) & (cy > 4) & (cy < 40)

# compare our brightness with the catalogue's, star by star
offset = {}
for b, col in ((1, "w1mpro"), (2, "w2mpro")):
    ours = ZP[b] - 2.5 * np.log10([ap_sum(template[b], x, y) for x, y in zip(cx[use], cy[use])])
    d = np.array(cat[col])[use] - ours
    d = d[np.isfinite(d)]
    _, offset[b], spread = sigma_clipped_stats(d)
    print(f"W{b}: {len(d)} stars, catalogue minus ours = {offset[b]:+.2f} mag (spread {spread:.2f})")

# convert counts to mJy, and add the neighbour star's extra noise to the error bars
F0 = {1: 309.540, 2: 171.787}          # Jy of a zero-magnitude source in W1 and W2
extra = {}
for b in (1, 2):
    s = lc2[(lc2["band"] == b) & (lc2["visit"] < 12)]
    extra[b] = np.sqrt(max(np.std(s["flux"]) ** 2 - np.median(s["err"]) ** 2, 0))
per_count = np.array([F0[b] * 1e3 * 10 ** (-0.4 * (ZP[b] + offset[b])) for b in lc2["band"]])
err_tot = np.sqrt(np.array(lc2["err"]) ** 2 + np.array([extra[b] for b in lc2["band"]]) ** 2)
lc2["mJy"] = np.array(lc2["flux"]) * per_count
lc2["mJy_err"] = err_tot * per_count
lc2.write("/content/barbie_lightcurve_mJy.csv", overwrite=True)

plt.figure(figsize=(9, 5))
for b, colour, wl in ((1, "tab:blue", "3.4"), (2, "tab:red", "4.6")):
    s = lc2[lc2["band"] == b]
    plt.errorbar(Time(s["mjd"], format="mjd").decimalyear, s["mJy"], s["mJy_err"],
                 fmt="o", color=colour, label=f"W{b} ({wl} µm)")
plt.axhline(0, color="grey", lw=0.8)
plt.axvline(2020.46, color="k", ls=":", label="optical flare starts (June 2020)")
plt.xlabel("Year")
plt.ylabel("Brightness above the 'before' level (mJy)")
plt.legend()
plt.show()

In [ ]:
# Scary Barbie, step 4b: dust temperature, power and size at each flare visit
from astropy import constants as const
from astropy.cosmology import Planck18

z = 0.995
h, c, k, sig = const.h.value, const.c.value, const.k_B.value, const.sigma_sb.value
nu = {1: c / 3.368e-6 * (1 + z), 2: c / 4.618e-6 * (1 + z)}   # frequencies the light left with
DL = Planck18.luminosity_distance(z).to("m").value

def B(nu_, T):                                   # blackbody brightness (SI units)
    return 2 * h * nu_**3 / c**2 / np.expm1(h * nu_ / (k * T))

Tgrid = np.linspace(500, 8000, 3000)
ratio_grid = B(nu[2], Tgrid) / B(nu[1], Tgrid)  # W2/W1 ratio for each temperature
rng = np.random.default_rng(1)

def get(v, b, col):
    return float(lc2[(lc2["visit"] == v) & (lc2["band"] == b)][col][0])

print("  year    T (K)               log L (erg/s)   R (pc)")
for v in range(13, 21):
    d1 = rng.normal(get(v, 1, "mJy"), get(v, 1, "mJy_err"), 4000)
    d2 = rng.normal(get(v, 2, "mJy"), get(v, 2, "mJy_err"), 4000)
    ok = (d1 > 0) & (d2 > 0)
    T = np.interp(d2[ok] / d1[ok], ratio_grid[::-1], Tgrid[::-1])
    R2 = d1[ok] * 1e-29 * DL**2 / ((1 + z) * np.pi * B(nu[1], T))   # glowing area, from W1
    L = 4 * np.pi * R2 * sig * T**4 * 1e7                           # watts -> erg/s
    lo, mid, hi = np.percentile(T, [16, 50, 84])
    yr = Time(get(v, 1, "mjd"), format="mjd").decimalyear
    print(f"{yr:7.2f}  {mid:5.0f} (-{mid - lo:4.0f} +{hi - mid:4.0f})   "
          f"{np.log10(np.median(L)):6.2f}          {np.sqrt(np.median(R2)) / 3.086e16:.3f}")

In [ ]:
# Scary Barbie, step 5a: the optical light curve from ZTF, via the public ALeRCE broker
import requests

r = requests.get("https://api.alerce.online/ztf/v1/objects/ZTF20abrbeie/lightcurve", timeout=120)
r.raise_for_status()
opt = Table(rows=[(d["mjd"], d["fid"], d["magpsf"], d["sigmapsf"]) for d in r.json()["detections"]],
            names=("mjd", "fid", "mag", "err"))
opt = opt[(opt["fid"] == 1) | (opt["fid"] == 2)]              # 1 = g band, 2 = r band
opt["mJy"] = 3.631e6 * 10 ** (-0.4 * np.array(opt["mag"]))    # AB magnitudes -> mJy
opt["mJy_err"] = opt["mJy"] * np.array(opt["err"]) / 1.0857
opt.write("/content/barbie_ztf_alerce.csv", overwrite=True)
print(len(opt), "ZTF detections, from", Time(opt["mjd"].min(), format="mjd").iso[:10],
      "to", Time(opt["mjd"].max(), format="mjd").iso[:10])

plt.figure(figsize=(10, 5))
for fid, colour, name in ((1, "tab:green", "ZTF g"), (2, "tab:orange", "ZTF r")):
    s = opt[opt["fid"] == fid]
    plt.errorbar(Time(s["mjd"], format="mjd").decimalyear, s["mJy"], s["mJy_err"],
                 fmt=".", color=colour, alpha=0.6, label=name)
for b, colour, wl in ((1, "tab:blue", "3.4"), (2, "tab:red", "4.6")):
    s = lc2[(lc2["band"] == b) & (lc2["visit"] >= 12)]
    plt.errorbar(Time(s["mjd"], format="mjd").decimalyear, s["mJy"], s["mJy_err"],
                 fmt="o", color=colour, label=f"NEOWISE W{b} ({wl} µm)")
plt.xlabel("Year")
plt.ylabel("Brightness (mJy)")
plt.legend()
plt.show()

In [ ]:
# Scary Barbie, step 5b: remove the flare's own light from W1 and W2, then refit the dust
EBV = 0.12                                           # Milky Way reddening (Subrayan et al. 2023)
R_ext = {"g": 3.7, "r": 2.6, 1: 0.18, 2: 0.16}       # extinction per unit E(B-V), each band
nu_opt = {"g": c / 4746e-10 * (1 + z), "r": c / 6366e-10 * (1 + z)}
N = 4000

def binned(fid):                                     # ZTF light curve in 10-day medians
    s = opt[opt["fid"] == fid]
    idx = np.digitize(np.array(s["mjd"]), np.arange(s["mjd"].min(), s["mjd"].max() + 10, 10))
    t = np.array([np.mean(s["mjd"][idx == i]) for i in np.unique(idx)])
    f = np.array([np.median(s["mJy"][idx == i]) for i in np.unique(idx)])
    return t, f
tg, fg = binned(1)
tr, fr = binned(2)

Topt_grid = np.linspace(5000, 60000, 5000)
gr_grid = B(nu_opt["g"], Topt_grid) / B(nu_opt["r"], Topt_grid)    # g/r ratio vs temperature

def flare_bb(mjd):
    """The flare's temperature, power, and light at W1 and W2 on a date (N random draws)."""
    g = 10 ** np.interp(mjd, tg, np.log10(fg)) * 10 ** (0.4 * R_ext["g"] * EBV) * rng.normal(1, 0.05, N)
    r = 10 ** np.interp(mjd, tr, np.log10(fr)) * 10 ** (0.4 * R_ext["r"] * EBV) * rng.normal(1, 0.05, N)
    T = np.interp(g / r, gr_grid, Topt_grid)
    R2 = r * 1e-29 * DL**2 / ((1 + z) * np.pi * B(nu_opt["r"], T))
    L = 4 * np.pi * R2 * sig * T**4 * 1e7
    return T, L, r * B(nu[1], T) / B(nu_opt["r"], T), r * B(nu[2], T) / B(nu_opt["r"], T)

cal1 = 10 ** (-0.4 * rng.normal(0, 0.07, N))         # W1 calibration uncertainty
cal2 = 10 ** (-0.4 * rng.normal(0, 0.04, N))         # W2 calibration uncertainty
ext1, ext2 = 10 ** (0.4 * R_ext[1] * EBV), 10 ** (0.4 * R_ext[2] * EBV)

T0, Lopt_peak, _, _ = flare_bb(59291)                # optical peak, 2021 March 18
print(f"optical peak: T = {np.median(T0):.0f} K, log L = {np.log10(np.median(Lopt_peak)):.2f}\n")

print("  year   flare T  flare share of W1   dust T (K)            log L_dust   R (pc)")
Ldust = {}
for v in range(13, 21):
    mjd = get(v, 1, "mjd")
    Tf, Lf, b1, b2 = flare_bb(mjd)
    m1 = rng.normal(get(v, 1, "mJy"), get(v, 1, "mJy_err"), N) * cal1 * ext1
    m2 = rng.normal(get(v, 2, "mJy"), get(v, 2, "mJy_err"), N) * cal2 * ext2
    d1, d2 = m1 - b1, m2 - b2                        # dust light only
    ok = (d1 > 0) & (d2 > 0)
    T = np.full(N, np.nan)
    T[ok] = np.interp(d2[ok] / d1[ok], ratio_grid[::-1], Tgrid[::-1])
    R2 = d1 * 1e-29 * DL**2 / ((1 + z) * np.pi * B(nu[1], T))
    Ldust[v] = 4 * np.pi * R2 * sig * T**4 * 1e7
    lo, mid, hi = np.nanpercentile(T, [16, 50, 84])
    yr = Time(mjd, format="mjd").decimalyear
    print(f"{yr:7.2f}  {np.median(Tf):6.0f}     {np.median(b1 / m1):5.0%}          "
          f"{mid:5.0f} (-{mid - lo:4.0f} +{hi - mid:4.0f})    {np.log10(np.nanmedian(Ldust[v])):6.2f}"
          f"      {np.sqrt(np.nanmedian(R2)) / 3.086e16:.3f}")

peak = max(Ldust, key=lambda v: np.nanmedian(Ldust[v]))
lo, mid, hi = np.nanpercentile(Ldust[peak] / Lopt_peak, [16, 50, 84])
print(f"\ndust is brightest at visit {peak}; covering fraction = {mid:.2f} (-{mid - lo:.2f} +{hi - mid:.2f})")
print("published: Hinkle et al. 2025 = 0.42 ± 0.12;  Wiseman et al. 2025 = 0.06")

In [ ]:
# ===== Step 6: Figure 1 + results table (run after steps 1-5b, same session) =====
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.time import Time

OUT = "/content"
rng = np.random.default_rng(2026)               # fixed seed, so every run gives the same numbers
cal1 = 10 ** (-0.4 * rng.normal(0, 0.07, N))    # W1 calibration uncertainty
cal2 = 10 ** (-0.4 * rng.normal(0, 0.04, N))    # W2 calibration uncertainty

def colour_T(f1, f2):                           # dust temperature from W1, W2 fluxes
    T = np.full(N, np.nan)
    ok = (f1 > 0) & (f2 > 0)
    T[ok] = np.interp(f2[ok] / f1[ok], ratio_grid[::-1], Tgrid[::-1])
    return T

def pct(x):                                     # 16th, 50th, 84th percentiles
    return list(np.nanpercentile(x, [16, 50, 84]))

def year(mjd):
    return Time(np.asarray(mjd, dtype=float), format="mjd").decimalyear

# ---- 1. The numbers for each flare visit ----
rows, Ld = [], {}
for v in range(13, 21):
    mjd = get(v, 1, "mjd")
    Tf, Lf, b1, b2 = flare_bb(mjd)              # the flare's own light in W1 and W2
    m1 = rng.normal(get(v, 1, "mJy"), get(v, 1, "mJy_err"), N) * cal1 * ext1
    m2 = rng.normal(get(v, 2, "mJy"), get(v, 2, "mJy_err"), N) * cal2 * ext2
    T_raw = colour_T(m1, m2)                    # before removing the flare's light
    d1 = m1 - b1
    T = colour_T(d1, m2 - b2)                   # after removing it
    R2 = d1 * 1e-29 * DL**2 / ((1 + z) * np.pi * B(nu[1], T))
    Ld[v] = 4 * np.pi * R2 * sig * T**4 * 1e7
    rows.append([v, float(year(mjd)), mjd, get(v, 1, "mJy"), get(v, 1, "mJy_err"),
                 get(v, 2, "mJy"), get(v, 2, "mJy_err"), np.median(Tf), np.median(b1) / np.median(m1)]
                + pct(T_raw) + pct(T) + pct(np.log10(Ld[v])) + pct(np.sqrt(R2) / 3.0857e16))

names = ["visit", "year", "mjd", "W1_mJy", "W1_err", "W2_mJy", "W2_err", "flare_T_K", "flare_share_W1",
         "Traw_16", "Traw_50", "Traw_84", "T_16", "T_50", "T_84",
         "logL_16", "logL_50", "logL_84", "R_pc_16", "R_pc_50", "R_pc_84"]
res = Table(rows=rows, names=names)
for n in names[1:]:
    res[n].format = {"y": ".3f", "m": ".2f", "W": ".4f", "f": ".3f", "T": ".0f", "l": ".2f", "R": ".4f"}[n[0]]
res["flare_T_K"].format = ".0f"
res.write(f"{OUT}/barbie_results.csv", overwrite=True)

peak = max(Ld, key=lambda v: np.nanmedian(Ld[v]))
_, Lopt, _, _ = flare_bb(59291)                 # optical luminosity at the optical peak
fc = pct(Ld[peak] / Lopt)
print(res["year", "flare_share_W1", "Traw_50", "T_16", "T_50", "T_84", "logL_50", "R_pc_50"])
print(f"\nDust brightest at {res['year'][peak - 13]:.2f}: log L = {res['logL_50'][peak - 13]:.2f}, "
      f"covering fraction = {fc[1]:.2f} (-{fc[1] - fc[0]:.2f} +{fc[2] - fc[1]:.2f})")

# ---- 2. The figure ----
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, WHITE = "#e1e0d9", "#c3c2b7", "#ffffff"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({
    "font.size": 10, "axes.edgecolor": AXIS, "axes.labelcolor": INK,
    "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": False, "figure.facecolor": WHITE, "axes.facecolor": WHITE})

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6.2), sharex=True,
                               gridspec_kw={"height_ratios": [1.1, 1], "hspace": 0.08})
for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)

# (a) light curves: NEOWISE W1, W2 and ZTF r
hand = {}
for b, col, mk, ms in [(2, ORANGE, "s", 8), (1, BLUE, "o", 6.5)]:    # W2 first, smaller W1 on top
    s = lc2[lc2["band"] == b]
    hand[b] = ax1.errorbar(year(s["mjd"]), s["mJy"], yerr=s["mJy_err"], fmt=mk, ms=ms, color=col,
                           mec=WHITE, mew=1.5, elinewidth=1, capsize=0, zorder=5 - b)
gap = np.where(np.diff(tr) > 60)[0] + 1         # break the line where ZTF could not observe
hand["r"], = ax1.plot(np.insert(year(tr), gap, np.nan), np.insert(fr, gap, np.nan), color=AQUA,
                      lw=1.5, solid_joinstyle="round", solid_capstyle="round", zorder=2)
j = int(np.argmin(np.abs(tr - (tr[np.argmax(fr)] + 250))))     # a point on the decline
ax1.annotate("ZTF r", (float(year(tr[j])), fr[j]), xytext=(-3, -5), textcoords="offset points",
             ha="right", va="top", color=INK2, fontsize=9)
ax1.axhline(0, color=AXIS, lw=0.8, zorder=1)
ax1.yaxis.set_major_locator(plt.MultipleLocator(0.1))
ax1.set_ylabel("Flux density (mJy)")
ax1.legend([hand[1], hand[2], hand["r"]], ["NEOWISE W1 (3.4 µm)", "NEOWISE W2 (4.6 µm)", "ZTF r (optical)"],
           loc="upper left", fontsize=9)

# (b) dust temperature, before and after removing the flare's light
x = np.array(res["year"])
ax2.axhspan(1500, 2000, color=MUTED, alpha=0.15, lw=0, zorder=0)
ax2.text(2017.75, 1750, "dust sublimation (1500–2000 K)", va="center", color=INK2, fontsize=9)
ax2.plot(x, np.array(res["Traw_50"]), "_", ms=14, mew=2, color=MUTED,
         label="before removing flare light", zorder=2)
ax2.errorbar(x, np.array(res["T_50"]),
             yerr=[np.array(res["T_50"] - res["T_16"]), np.array(res["T_84"] - res["T_50"])],
             fmt="o", ms=7, color=INK, mec=WHITE, mew=1.5, elinewidth=1, capsize=0,
             label="after removing flare light", zorder=3)
ax2.set_ylim(1000, 8000)
ax2.set_yticks([2000, 4000, 6000, 8000])
ax2.set_xlim(2017.6, 2024.8)
ax2.set_ylabel("Dust color temperature (K)")
ax2.set_xlabel("Year")
ax2.legend(loc="upper left", fontsize=9)

fig.savefig(f"{OUT}/barbie_figure1.png", dpi=300, bbox_inches="tight")
plt.show()

try:                                            # download both files to your computer
    from google.colab import files
    files.download(f"{OUT}/barbie_figure1.png")
    files.download(f"{OUT}/barbie_results.csv")
except ImportError:
    pass